# WXH · Experiment 2 · Connectivity-only GCN (4 bands x PLV / |ciPLV|) (historical wMNE graphs, frozen protocol)

The existing D3 definition (`configs/experiments/d3/d3_exp2_conn_minimal_fixedhp.yaml`, unchanged) on the existing
wMNE store `graphsD3_v4_*` (20 s windows, 50 % overlap, bands theta / alpha1 / alpha2 / beta), evaluated with the
frozen LCMV / Xu protocol (same 52 / 13 participants and five development folds, one repeat) instead of `within_D3`.
**Validation-matched only:** windows and bands differ from the LCMV suite, so this does not isolate the inverse;
it separates the effect of the validation protocol from the earlier D3 results. - **8 variant(s):** `plv__theta`, `plv__alpha1`, `plv__alpha2`, `plv__beta`, `ciplv__theta`, `ciplv__alpha1`, `ciplv__alpha2`, `ciplv__beta`
- **Settings:** xu_dev_cv, xu_holdout (cross-dataset only if the DESIGN.md §7 gate passes)
- **Protocol (from the config, unchanged):** 1 repeats × 5-fold subject-level outer CV; fixed hyperparameters, no hyperparameter search and no inner CV: one fit per outer training fold (protocol `gnn_fixed_hparams_v1`); 60 epochs.

**How to use:** *Runtime → Run all*. Nothing to edit. Allow Google Drive access when asked.

If Colab disconnects or the runtime ends, open the notebook again and *Run all* again: finished
work is kept on Google Drive and skipped, and the computation continues where it stopped.
Everything (checkpoints, results, figures, manifests) is written to Drive, never to the
temporary Colab disk.

**Suite:** matched wMNE counterpart of the D3 LCMV / Xu suite (methodology log entry 55, `docs/wmne_xu.md`). All
outputs are under `<outputs>/D3_WMNE_XU/`, with its own code pin `D3_WMNE_XU/CODE_COMMIT.txt`. The LCMV suite
(`D3_LCMV_XU/`: stores, results, pins), `graphsD3_v4_*` and D1/D2 are only read, never written. Safe to run while
LCMV experiments run in another Colab session (separate folders and pin; one GPU per session).
One mode (all windows of the historical store). Results: `<outputs>/D3_WMNE_XU/experiments_H_historical_graphs/`. Use a **GPU runtime** (T4).

In [ ]:
# ===== D3 wMNE / Xu suite setup: nothing to edit. Mounts Drive, gets the code, installs the pinned environment. =====
UPDATE_WX_CODE = False   # leave False. True only if you were asked to switch this suite to newer code
# The wMNE / Xu suite has its own code pin, <outputs>/D3_WMNE_XU/CODE_COMMIT.txt (all WX notebooks run
# that commit); the LCMV suite pin D3_LCMV_XU/CODE_COMMIT.txt and all other pins are only reported.
BRANCH = 'claude/ecstatic-davinci-a2e8mh'
REPO_URL = 'https://github.com/Samimj9/Source-space-EEG-graph-learning-for-Alzheimer-s-disease-using-multi-band-functional-connectivity.git'

import os, subprocess, sys
from google.colab import drive
drive.mount('/content/drive')

def _run(cmd, cwd=None):  # shows output and stops "Run all" on failure
    p = subprocess.run(cmd, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    print(p.stdout)
    if p.returncode:
        raise RuntimeError(f'failed: {" ".join(cmd)}')

os.chdir('/content')
_run(['rm', '-rf', '/content/repo'])
_run(['git', 'clone', '-q', '-b', BRANCH, REPO_URL, '/content/repo'])
os.chdir('/content/repo')
_run([sys.executable, 'scripts/colab_setup.py', '--family', 'WX']
     + (['--update-code'] if UPDATE_WX_CODE else []))
sys.path.insert(0, '/content/repo/src')   # the package, for this notebook's own cells

from adgraph.colab import sh
from adgraph.config import load_paths
PATHS = load_paths('configs/paths.yaml')
OUT = PATHS['processed_dir']
print('outputs on Drive:', OUT)
print('Dataset 3 folder:', PATHS['dataset3']['root'])
LX = OUT + '/D3_LCMV_XU'
WX = OUT + '/D3_WMNE_XU'
print('suite outputs:', WX, '| LCMV reference (read only):', LX)

## 1. Status

In [ ]:
CONFIG = 'configs/experiments/wx_hist/wxh_exp2_conn_minimal_fixedhp.yaml'
sh(f"python scripts/experiment.py status --config {CONFIG}")

## 2. Run (resumable)

In [ ]:
ERROR = None
try:
    sh(f"python scripts/experiment.py run --config {CONFIG}")
except RuntimeError as exc:
    ERROR = exc

## 3. Results (saved outputs only)

In [ ]:
import glob
from pathlib import Path
import pandas as pd
sh(f"python scripts/experiment.py report --config {CONFIG}")
reports = sorted(glob.glob(f'{WX}/experiments_H_historical_graphs/{Path(CONFIG).stem}/_reports/report_*'))
if reports:
    display(pd.read_csv(reports[-1] + '/results_summary.csv'))
if ERROR:
    raise RuntimeError('some units failed; completed units are saved. Run all again.')